# Object enrichments in ApertureDB: querying what the detectors found

Google Cloud Storage's **object enrichments** run detectors (labels, objects, faces, content safety) on the
images in a bucket. `ingest.py` and `enrich.py` copied the images and those detections into ApertureDB
as a graph; this notebook queries it.

**Contents:** 1 What is in the data (labels, safety ratings) - 2 The headline question - 3 Family friendly (content safety) - 4 String search - 5 Expressions - 6 Graph traversal - 7 Graph views - 8 Spatial - 9 Detector cut-off. Every query has a title, a line on what it shows, and its JSON is printed above its result.

```
 _Image ──HasLabel──────▶ GCSLabel        50 whole-image labels with a score
   ▲  └──HasDetection──▶ _BoundingBox      same boxes, with the edge properties detector / score / stored_from
   └───── _BoundingBox                    object boxes (named by the object) and face boxes (label "face"),
                                          each with its own properties (score, joy, pan angle, ...)
```

Everything here is **phase 1**: structured metadata only (embeddings and similarity search come next).
The queries use ApertureDB's string operators, property expressions and graph traversal. The same queries
are saved as JSON in `queries/` so you can paste them into the ApertureDB web UI.

Run `./setup.sh` and `./demo.sh --bucket ... --prefix ...` first (see the README).

## Connect

In [ ]:
import html
import json
import os

import pandas as pd
from IPython.display import HTML, display

import config          # which ApertureDB: local Community Edition, or APERTUREDB_KEY / .env
import graph_viz as gv  # interactive graph views
import queries         # every query below, as plain functions returning ApertureDB queries

client = config.get_client()

DATASET = None  # set to "<bucket>/<prefix>" to restrict every query to one ingested folder
LABELS = {"_Image": "gcs_object_stem", "GCSLabel": "description"}   # how graph nodes are named
BY_SCORE = {"GCSLabel": ("score", "descending")}                    # keep the best labels first

In [ ]:
def run(query):
    """Run a query; raise if the server rejects a command; return (response, blobs)."""
    response, blobs = client.query(query)
    if isinstance(response, dict):
        raise RuntimeError(response.get("info"))
    for r in response:
        body = next(iter(r.values()))
        if body.get("status", 0) not in (0, 2):
            raise RuntimeError(body.get("info"))
    return response, blobs


def table(response, index=-1, columns=None):
    """The entities one command returned, as a DataFrame."""
    body = next(iter(response[index].values()))
    if "groups" in body:  # results.group: one row per group, with its count (and any avg/min/max)
        df = pd.DataFrame(body["groups"]).drop(columns="_group_class", errors="ignore")
        if "_group_avg" in df:
            df["avg_score"] = df.pop("_group_avg").map(lambda d: round(d["score"], 2))
        return df.rename(columns={"_group_count": "count"}).sort_values("count", ascending=False, ignore_index=True)
    rows = body.get("entities", [])
    if isinstance(rows, dict):  # grouped by source
        rows = [e for group in rows.values() for e in group]
    df = pd.DataFrame(rows)
    return df if columns is None or df.empty else df[[c for c in columns if c in df]]


def thumbs(blobs, captions, width=170):
    """Show returned pixels (posters or box crops) in a row."""
    import base64
    cells = "".join(
        f'<figure style="display:inline-block;margin:4px;text-align:center;font:12px sans-serif">'
        f'<img width="{width}" src="data:{gv.sniff_mime(b)};base64,{base64.b64encode(b).decode()}">'
        f'<figcaption>{html.escape(str(c))}</figcaption></figure>' for b, c in zip(blobs, captions))
    display(HTML(cells))


def compact(obj, indent=0, width=96):
    """JSON text that keeps short pieces on one line, so a query stays readable."""
    flat = json.dumps(obj)
    if len(flat) + indent <= width or not isinstance(obj, (dict, list)):
        return flat
    pad = " " * (indent + 2)
    if isinstance(obj, dict):
        items = [f"{pad}{json.dumps(k)}: {compact(v, indent + 2, width)}" for k, v in obj.items()]
        return "{\n" + ",\n".join(items) + "\n" + " " * indent + "}"
    return "[\n" + ",\n".join(pad + compact(v, indent + 2, width) for v in obj) + "\n" + " " * indent + "]"


def show(query, caption=None, columns=None, top=None):
    """Print the query, run it, show the rows (the first ``top``) and - if it returned pixels - the pictures."""
    display(HTML("<details open><summary><b>Query</b> (ApertureDB JSON)</summary><pre>"
                 + html.escape(compact(query)) + "</pre></details>"))
    response, blobs = run(query)
    df = table(response, columns=columns)
    if not len(df):
        display(HTML("<i>no rows - ingest more data or loosen the filter</i>"))
    else:
        display(df.head(top) if top else df)
        if top and len(df) > top:
            display(HTML(f"<i>showing {top} of {len(df)} rows</i>"))
    if blobs and caption and len(df):
        thumbs(blobs, [caption(r) for _, r in df.iterrows()])
    return response, blobs

## The data model

Live from the database: one node per class (with how many objects it holds), one edge per connection class.
Hover a class to highlight its connections, click it for its properties and a query that lists it.

In [ ]:
gv.render_schema(client)

## 1. What is in the data?

Before asking questions, see what Google's detectors actually found. `results.group` aggregates inside the database: no rows to download, one row per group.

### 1.1 Which labels did Google find, and how many posters carry each?

**What it shows:** the label vocabulary of the whole dataset: every distinct label with its poster count and average score. Use it to pick words for the queries below.

`results.group` on `description` plus `count` and `avg`, over labels scoring at least 0.5. (The server groups first and orders afterwards, so the table is sorted here.)

In [ ]:
response, _ = show(queries.label_vocabulary(min_score=0.5, dataset=DATASET), top=15)
vocabulary = table(response)
print(f"{len(vocabulary):,} distinct labels")

### 1.2 Is a particular word in the vocabulary?

**What it shows:** a quick lookup in the table above, e.g. is there an 'alien', a 'robot', a 'ghost'?

Plain pandas over the result of 1.1; change the words.

In [ ]:
vocabulary[vocabulary.description.str.contains("alien|robot|ghost|superhero|monster", case=False)]

### 1.3 How did the content-safety detector rate the posters?

**What it shows:** the number of posters for every (adult, violent) rating pair; the scale is VERY_UNLIKELY, UNLIKELY, POSSIBLE, LIKELY, VERY_LIKELY.

`results.group` on two properties at once.

In [ ]:
_ = show(queries.safety_ratings(dataset=DATASET), top=12)

## 2. The headline question: movies about fictional characters where everyone looks happy

### 2.1 A label, every face smiling, one request

**What it shows:** posters labelled *fictional character* on which **every** detected face has joy of at least 0.7. Posters without faces never match.

Label search (`icontains`), a hop from the label to its poster, and a condition on `min_face_joy` (the least happy face on the poster, stored at load time) - in one query. Try `term="cartoon"` or `"superhero"`.

In [ ]:
_ = show(queries.everyone_looks_happy(dataset=DATASET),
         caption=lambda r: f"{r.face_count} face(s), least happy {r.min_face_joy:.2f}",
         columns=["gcs_object_name", "face_count", "min_face_joy", "label_count"])

## 3. Family friendly

The content-safety detector rates every poster for adult and violent content (VERY_UNLIKELY ... VERY_LIKELY). Combined with labels and faces from the other detectors, one request can answer a natural-language ask such as *"something animated for the whole family where everyone looks happy"*.

### 3.1 Family movie night: a natural-language ask as one query

**What it shows:** family-friendly (both safety ratings VERY_UNLIKELY) posters labelled *animation* on which every face is smiling.

Three detectors in one request: a label search, a hop to the poster, then conditions on the safety ranks and on `min_face_joy`. Try `term="cartoon"` or `min_joy=0.8`.

In [ ]:
_ = show(queries.family_movie_night(dataset=DATASET),
         caption=lambda r: f"{r.face_count} face(s), least happy {r.min_face_joy:.2f}",
         columns=["gcs_object_name", "face_count", "min_face_joy", "adult_content_likelihood", "violent_content_likelihood"])

### 3.2 Family-friendly posters for any label

**What it shows:** the same filter without the faces: posters labelled *animation* that are rated VERY_UNLIKELY adult **and** VERY_UNLIKELY violent.

A label search, a graph hop, and two rank conditions combined with `all`. Try another word from 1.1.

In [ ]:
_ = show(queries.family_friendly("animation", dataset=DATASET),
         caption=lambda r: f"{r.adult_content_likelihood} / {r.violent_content_likelihood}")

### 3.3 Posters a family catalogue should hold back

**What it shows:** the other side of the filter: either rating at least POSSIBLE (rank 3 of 5).

An `any` expression across two properties written by the same detector; no pictures, just the ratings.

In [ ]:
_ = show(queries.sensitive_content(dataset=DATASET))

### 3.4 Why was a poster held back?

**What it shows:** the held-back posters together with their highest-scoring labels, which usually explain the rating.

Two commands: the flagged images, then their connected labels sorted by score.

In [ ]:
_ = show(queries.why_flagged(dataset=DATASET), columns=['description', 'score', 'image_id'])

## 4. String search

Operators: `contains`, `starts_with`, `ends_with`, `like` (`%` and `_`), `regex` (RE2, `(?i)` for any case) and the case-insensitive `i...` versions. Trigram indexes on these properties keep them fast.

### 4.1 Labels containing a word, then their posters

**What it shows:** a case-insensitive substring search over label text (`icontains`), then a hop to the posters that carry the matching labels.

Change `"action"` to any word from 1.1.

In [ ]:
_ = show(queries.labels_containing("action", min_score=0.6, dataset=DATASET, images=True, limit=8),
         caption=lambda r: r.gcs_object_name.rsplit("/", 1)[-1])

### 4.2 Object names matching a regular expression

**What it shows:** posters with an object box named man, woman, girl or boy: the object detector's person-like names.

`regex` with `(?i)` for case-insensitive.

In [ ]:
_ = show(queries.people_by_name(dataset=DATASET, limit=10))

### 4.3 Combining words: any, none, and a score

**What it shows:** labels containing *action* or *fiction* but not *game*, above a score, best first.

An array expression with `any`, `not` and a numeric condition. Try `golf` or `romance`.

In [ ]:
_ = show(queries.label_search_boolean(include=("action", "fiction"), exclude=("game",), dataset=DATASET),
         columns=["description", "score", "image_id"])

### 4.4 Pattern shapes

**What it shows:** labels ending in *ing* (`like '%ing'`) or starting with *for* (`istarts_with`).

Two patterns inside one `any`.

In [ ]:
_ = show(queries.label_shapes(dataset=DATASET), columns=['description', 'score'])

### 4.5 Word search with typo tolerance

**What it shows:** a full-text search over label text: `animatd` (a typo) still finds *Animation* and *Animated cartoon*.

A `fulltext` index on the label text. It scores whole words, ignores case and allows `max_edits` typos per word. Unlike `icontains` it does not match inside words.

In [ ]:
_ = show(queries.label_word_search("animatd cartoon", max_edits=1, dataset=DATASET), columns=["gcs_object_name", "label_count", "face_count"])

## 5. Expressions: compare and compute on properties, inside the database

### 5.1 Happy faces looking at the camera

**What it shows:** faces looking at the camera (`abs(pan_angle) < 20`) that show more joy than sorrow **and** more joy than anger, as cropped faces.

`$joy > $sorrow` compares two properties of the same face and nothing is post-processed in Python. The faces are stored with their pixels, so the request returns the crops.

In [ ]:
_ = show(queries.happy_frontal_faces(dataset=DATASET, crops=True, limit=8),
         caption=lambda r: f"joy {r.joy:.2f}", columns=["image_id", "joy", "pan_angle", "area_fraction"])

### 5.2 Delighted rather than just happy

**What it shows:** faces where `joy + surprise > 0.85` while `sorrow < 0.1`.

Arithmetic over properties.

In [ ]:
_ = show(queries.delighted_faces(dataset=DATASET))

### 5.3 Posters dominated by a single face

**What it shows:** a confidently detected face that covers a large share of the poster (`area_fraction`).

Numeric conditions on box properties; the crops are returned.

In [ ]:
_ = show(queries.dominant_faces(dataset=DATASET, crops=True),
         caption=lambda r: f"{r.area_fraction:.0%} of poster", columns=["image_id", "area_fraction", "detection_confidence"])

### 5.4 Data-quality check: images without results

**What it shows:** how many images have no face-detector result stored yet (zero when everything is enriched).

`is_missing` on a property.

In [ ]:
response, _ = run(queries.not_yet_enriched(dataset=DATASET))
response[0]["FindImage"]["count"]

## 6. Graph traversal

### 6.1 Label, poster, then that poster's happy faces

**What it shows:** three hops in one request for posters labelled *film*: label -> poster -> its faces with joy of at least 0.3.

Chained `_ref` / `is_connected_to` / `image_ref`.

In [ ]:
_ = show(queries.happy_faces_on_tagged_posters(term="film", min_joy=0.3, dataset=DATASET),
         columns=["image_id", "joy", "area_fraction"])

### 6.2 Posters connected to a label and to an object

**What it shows:** posters labelled *clothing* that also have a *person* object box.

`is_connected_to` with `all`.

In [ ]:
_ = show(queries.label_and_object_together("clothing", "person", dataset=DATASET))

### 6.3 Crowded posters from stored counts

**What it shows:** posters with many faces and many objects, as a plain filter on counts stored at load time.

No joins: `face_count` and `object_count` live on the image.

In [ ]:
_ = show(queries.crowded_posters(min_faces=3, min_objects=5, dataset=DATASET))

### 6.4 Objects below the cut-off, found through an edge property

**What it shows:** object boxes that were stored but score below 0.5, listed through the `score` property of their `HasDetection` edge.

No re-ingestion is needed to change the cut-off: widen `low`/`high` and run again. (`is_connected_to` with `connection_class` and edge `constraints`.)

In [ ]:
_ = show(queries.objects_between(0.3, 0.5, dataset=DATASET), columns=["object_name", "score", "image_id"])

## 7. Graph views

Draw what a query returned, with its neighbours. Highlighted nodes are the answer; click any node for its
properties, its picture (with every stored box drawn over it) and the query that fetches it. Use the legend
to hide a class and the search box to find a node.

In [ ]:
gv.render_query(client, queries.happy_frontal_faces(dataset=DATASET, limit=5), title="happy frontal faces",
                label_props=LABELS, sort=BY_SCORE, per_class=4)

Start from one poster instead and walk outwards (`depth=2` also reaches the posters sharing its labels):

In [ ]:
response, _ = run(queries.crowded_posters(min_faces=1, dataset=DATASET, limit=1))
poster = table(response).gcs_object_name.iloc[0]
poster

In [ ]:
gv.render_neighborhood(client, "_Image", {"gcs_object_name": ["==", poster]}, depth=1, per_class=12,
                       label_props=LABELS, sort=BY_SCORE, title=poster)

## 8. Spatial: boxes from two different detectors

**What it shows:** `RegionIoU` computes the intersection-over-union between two sets of regions. Here: which detected **person**
boxes overlap which **face** boxes on the same poster? (Swap `"Person"` for `"Logo"` to find logos covering faces.)

In [ ]:
response, _ = run(queries.overlap_candidates("Person", dataset=DATASET, limit=5))
candidates = table(response)
candidates

In [ ]:
if candidates.empty:
    print("No poster with both a Person and a face box in this dataset.")
else:
    image_id = candidates["id"].iloc[0]
    response, _ = run(queries.object_face_overlap(image_id, "Person"))
    people, faces = table(response, 1), table(response, 2)
    iou = pd.DataFrame(response[-1]["RegionIoU"]["IoU"],
                       index=[f"Person (score {s:.2f})" for s in people.score],
                       columns=[f"face (joy {j:.2f})" for j in faces.joy])
    display(iou.style.format("{:.2f}").background_gradient(cmap="Blues", axis=None))

## 9. Is the object-detector cut-off sensible?

`enrich.py` stores object proposals scoring at least `--confidence-threshold` (default **0.3**); the per-image `object_count`
and the queries above use **0.5**. Boxes in between stay in the database with their score on the `HasDetection` edge (see 6.4), so the
cut-off can move without calling Google again. `enrich_stats.json` records what the detector returned in total. The table shows how
many proposals fall in each score range, how many were stored and how many count as confident. Re-run with a different floor
(`python enrich.py --only object --replace --confidence-threshold 0.2`) to store more.

In [ ]:
path = "enrich_stats.json"
if not os.path.exists(path):
    print("Run enrich.py first - it writes", path)
else:
    stats = json.load(open(path))
    cut = stats["confidence_threshold"]
    detector = stats["detectors"].get("object")
    if not detector:
        print("The last enrich run did not process the object detector.")
    else:
        hist = pd.DataFrame({
            "score range": [f"{i / 10:.1f} - {(i + 1) / 10:.1f}" for i in range(10)],
            "proposals": detector["score_histogram"],
        })
        hist["stored"] = [n if (i + 1) / 10 > cut else 0 for i, n in enumerate(detector["score_histogram"])]
        hist["confident (>= 0.5)"] = [n if i >= 5 else 0 for i, n in enumerate(detector["score_histogram"])]
        hist["share"] = (hist.proposals / max(1, hist.proposals.sum()) * 40).round().astype(int).map(lambda n: "#" * n)
        print(f"threshold {cut}: stored {detector['kept']} of {detector['seen']} proposals "
              f"({detector['kept'] / max(1, detector['seen']):.0%}) over {detector['images']} images")
        display(hist)

## 10. Vector search, with and without the metadata

Optional: needs CLIP embeddings (`pip install -r requirements-embeddings.txt`, then `python embed.py`). The search finds posters that *look like* the
ask; the filter adds what the detectors know. `demo.ipynb` shows this on one ask, `quality.ipynb` scores several.

In [ ]:
import vector_search as vs

ASK = "a family-friendly animated movie where everyone looks happy"
try:
    rows, blobs = vs.search(client, ASK, k=6, criteria=vs.family_criteria(), dataset=DATASET)
except Exception as exc:   # no embeddings yet, or torch not installed
    print("Vector search not available:", str(exc)[:200])
else:
    thumbs(blobs, [f"sim {r['similarity']:.2f}, {r['face_count']} face(s)" for r in rows])

## Try your own

* Edit the arguments above (`"golf"`, thresholds, ...) or write a new query function in `queries.py`.
* Every query is saved as JSON in `queries/` - paste one into the ApertureDB web UI's query editor
  (`docker compose --profile ui up -d` -> http://localhost:8088, login admin / admin).
* `python graph_viz.py node _Image --where gcs_object_stem=<name> --label _Image=gcs_object_stem --open`
  draws any poster from the command line.
* Narrow everything to one folder with `DATASET = "<bucket>/<prefix>"` in the first cell.